# 01-cleaning

See [project guide](../../README.md) and [data requirements](../../data/README.md) before execution. Workspace: `data/tweets/`. External inputs are not included. Run cells in order; model fitting and network collection are not run during repository checks.


In [ ]:
from pathlib import Path
import sys
import os
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'project_paths.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT))
from project_paths import workspace
os.chdir(workspace('tweets'))


In [ ]:
%pprint
import numpy as np 
import pandas as pd
import os
import time
import datetime

import matplotlib.pyplot as plt
import seaborn as sns; sns.set()

%config InlineBackend.figure_format = 'retina'
import warnings
# Keep warnings visible when checking the research environment.

In [ ]:
# Locate folder
os.chdir(workspace('tweets'))
path = workspace('tweets')      

# Overview

In [ ]:
%%time
tw_pitt =  pd.read_csv('Copy of Copy of Coronavirus [EN] [Scaled x4] [Pittsburgh].csv', skiprows=6)
tw_pitt['Date'] = pd.to_datetime(tw_pitt.Date).dt.date
tw_pitt['date'] = pd.to_datetime(tw_pitt.Date).dt.date
tw_pitt = tw_pitt.sort_values(['Date']).reset_index()

In [ ]:
tw_pitt.head()

By checking features one-by-one, I delete features that are not useful (either emtpy, or all with the same content).\

**Some thoughts:**\
1. Individual vs organizational account: maybe only individual account?
2. Author: top authors including WPXI, KDKA, TribLIVE, PittsburghPG
3. Candidate features for potential weighting factor: 
    - **Impact**
    - **Impressions** on Twitter is a total tally of all the times the Tweet has been seen. This includes not only the times it appears in a one of your followers’ timeline but also the times it has appeared in search or as a result of someone liking the Tweet.
    - **Twitter Followers**
    - **Twitter Reply Count**
    - **Retweets**
    - **Tweets** (what is this?)
    - **Reach (new)** (what is this?)
4. **Need to do: change emoji to the explanation**
5. Most Lat, Lng is 0, thus not good for tracking the exact location
6. How to incorporate media contents (e.g. pic, video) of a twitter

In [ ]:
# tw_pitt.drop(columns=['Query Name'], inplace=True)
tw_pitt = tw_pitt[tw_pitt.Domain=='twitter.com']
cols_to_be_dropped = ['Query Name', 'Language', 'Page Type', 
                      'Country Code', 'Continent Code', 'Continent', 'Country', 'City Code',
                      'Assignment', 'Avatar', 'Category Details', 'Checked', 'City', 'Display URLs',
                      'Facebook Author ID', 'Facebook Comments', 'Facebook Likes', 
                      'Facebook Role', 'Facebook Shares', 'Facebook Subtype',
                      'Instagram Comments', 'Instagram Followers', 'Instagram Following',
                      'Instagram Interactions Count', 'Instagram Likes', 'Instagram Posts',
                      'Media Filter', 'Priority', 'Starred', 'Status', 'Subtype',
                      'Total Monthly Visitors', 'Twitter Channel Role',
                      'Copyright', 'Linkedin Engagement', 'Linkedin Impressions',
                      'Page Type Name', 'Reddit Score', 'Region', 'Region Code']
cols_to_be_dropped = list(set(cols_to_be_dropped) & set(tw_pitt.columns))
tw_pitt.drop(columns=cols_to_be_dropped, inplace=True, errors="ignore")
print(tw_pitt.columns)

# Cleaning - DF

In [ ]:
def clean_df( df, nonempty_features ): 
    
    # nonempty_features = ['Date', 'Full Text']
    # show null values
    plt.figure(figsize = (15,5))
    sns.heatmap(df.isnull(), cbar = False, cmap = 'YlGnBu')
    plt.show()  # few data without summary
    
    a = df.isnull().apply(lambda x : sum(x[nonempty_features]) >= 1 , axis = 1)
    drop_ob_indexs = a[a==True].index
    print("There are ", len(drop_ob_indexs), " empty observations needed to be removed." )
    
    df = df.drop(drop_ob_indexs)
          
    # drop duplicated obs
    Obnum_before = df.shape[0]
    df = df[ ~ df.duplicated() ]
    Obnum_after = df.shape[0]
    df.index = range(df.shape[0]) 
    
    Num_of_dup = Obnum_before - Obnum_after
    print(" " )
    print("There are ", Num_of_dup, " duplicated values in the data set." ) 
    
    return df

In [ ]:
tw_pitt['is_retweet'] = tw_pitt.Title.fillna('').str.startswith('RT').astype(int)
tw_pitt.date.value_counts()
# .sample(5)

In [ ]:
tw_pitt[['Title', 'Full Text', 'is_retweet']].sample(5)

In [ ]:
%%time
tw_pitt = clean_df( tw_pitt, nonempty_features = ['Date', 'Full Text'] )

In [ ]:
plt.figure(figsize = (15,5))
sns.heatmap(tw_pitt.head(1000).isnull(), cbar = False, cmap = 'YlGnBu')
plt.show()  # few data without summary

In [ ]:
# drop useless columns
cols_to_be_dropped = ['Expanded URLs','Hashtags','Interest','Last Assignment Date','Media URLs',
                      'Professions','Short URLs', 'Thread Author', 'Thread Created Date','Thread URL',
                      'Twitter Channel Role','Twitter Reply to', 'Twitter Retweet of','Copyright','Mentioned Authors',
                      'Gender']
tw_pitt.drop(columns=cols_to_be_dropped, inplace=True, errors="ignore")

In [ ]:
plt.figure(figsize = (15,5))
sns.heatmap(tw_pitt.isnull(), cbar = False, cmap = 'YlGnBu')

# Cleaning - Text

In [ ]:
tw_pitt.Title.tolist()[:10]

In [ ]:
tw_pitt['Full Text'].tolist()[:10]

In [ ]:
import re
from tqdm import tqdm

from nltk.corpus import stopwords as stpw
from nltk.stem.wordnet import WordNetLemmatizer
from nltk import word_tokenize, pos_tag # nltk.download('averaged_perceptron_tagger')
 
# lemmatization
lemma = WordNetLemmatizer() 

# stop words
def update_stopwords():
    from nltk.corpus import stopwords as stpw
    stopwords = []
    path = workspace('tweets')
    with open(path+"stopwords_en.txt", "r") as f1:
        for stopword in f1.readlines():
            stopwords.append(stopword.strip('\n'))
        
    with open(path+"extra_stopwords_en.txt", "r") as f2:
        for stopword in f2.readlines():
            stopwords.append(stopword.strip('\n'))

    stopwords.extend(stpw.words('english'))
    stopwords = set(stopwords)
    return stopwords
stopwords = update_stopwords()

In [ ]:
# Cleaning functions
def replace_words(temp):
    # temp = tweet.lower()
    temp = temp.replace('covid 19' , 'covid19')
    temp = temp.replace('covid-19' , 'covid19')
    temp = temp.replace('hong kong' , 'hongkong')
    temp = temp.replace('wu han' , 'wuhan') 
    temp = temp.replace('tests' , 'test')
    temp = temp.replace('tested' , 'test')
    temp = temp.replace('testing' , 'test')
    temp = temp.replace('reopening' , 'reopen')
    temp = temp.replace('ppl' , 'people')
    temp = temp.replace('rt ' , '')
    return temp
def filter_words(temp):
    # temp = tweet.lower()  
    # fist filter
    temp = [w for w in temp.split(' ') if not w in stopwords]
    temp = " ".join(word for word in temp)
    # to avoid removing contractions in english
    temp = re.sub("'", "", temp) 
    # Removing hashtags and mentions
    temp = re.sub("@[A-Za-z0-9_]+","", temp)
    temp = re.sub("#[A-Za-z0-9_]+","", temp)
    # Removing links
    temp = re.sub(r"http\S+", "", temp)
    temp = re.sub(r"www.\S+", "", temp)
    # Removing punctuations
    temp = re.sub('[()!?]', ' ', temp)
    temp = re.sub(r'\[.*?\]',' ', temp)
    # Filtering non-alphanumeric characters
    temp = re.sub("[^a-z0-9]"," ", temp)
    return temp
def convert_postag(word2pos):
    # word2pos = ('eating', 'VBG')
    word = word2pos[0]
    tag = word2pos[1]
    
    if tag.startswith('J'):
        tag = 'a'
    elif tag.startswith('V'):
        tag = 'v'
    elif tag.startswith('N'):
        tag = 'n'
    elif tag.startswith('R'):
        tag = 'r'
    else:
        tag = 'skip'
        
    return (word,tag)
def filter_useful_pos(temp):
    wordlst = word_tokenize(temp)
    word2pos_lst = pos_tag(wordlst)
    word2pos_lst = [convert_postag(word2pos) for word2pos in word2pos_lst]
    word2pos_lst = [word2pos for word2pos in word2pos_lst if not word2pos[1] == 'skip']
    return word2pos_lst
def clean_tweet(tweet):
    # print(tweet)
    temp = tweet.lower()
    temp = filter_words(temp)
    temp = replace_words(temp)
    
    word2pos_lst = filter_useful_pos(temp)
    temp = [WordNetLemmatizer().lemmatize(word2pos[0],word2pos[1]) for word2pos in word2pos_lst ]
    # second filter
    temp = [w for w in temp if not w in stopwords]
    temp = " ".join(word for word in temp)
    # print(temp)
    return temp

In [ ]:
# Notes
'''
                            WordNetLemmatizer().lemmatize( word , pos: str = "n" )

                                        "n" for nouns (default),
                                        "v" for verbs, 
                                        "a" for adjectives, 
                                        "r" for adverbs 
                                        "s" for satellite adjectives.
'''
        
WordNetLemmatizer().lemmatize('eating','v') # eat
WordNetLemmatizer().lemmatize('eating')     # eating


'''
                            # Get pos 词性

                            from nltk import word_tokenize, pos_tag
                            # nltk.download('averaged_perceptron_tagger')

                            pos_tag(wordslst) 

'''

pos_tag(["eat"])

In [ ]:
%%time
cleaned_tweets = tw_pitt['Full Text'].apply(lambda x: clean_tweet(x))
tw_pitt['cleaned_text'] = cleaned_tweets

In [ ]:
# Furter clean
def filter_words2(temp):
    # temp = tweet.lower()  
    # fist filter
    temp = [w for w in temp.split(' ') if not w in stopwords]
    temp = [w for w in temp if len(w)>2]
    
    temp = " ".join(word for word in temp)
    return temp

In [ ]:
%%time
cleaned_tweets = tw_pitt['cleaned_text'].apply(lambda x: filter_words2(x))
tw_pitt['cleaned_text'] = cleaned_tweets
print('Total',(tw_pitt['cleaned_text'].apply(len)==0).sum(),'empty texts')

In [ ]:
drop_obs = tw_pitt[(tw_pitt['cleaned_text'].apply(len)==0)].index
tw_pitt = tw_pitt.drop(drop_obs).reset_index()
print('Total',(tw_pitt['cleaned_text'].apply(len)==0).sum(),'empty texts')

In [ ]:
tw_pitt.to_csv('tw_pitt_cleaned.csv', index=False)
tw_pitt[['Date', 'cleaned_text','is_retweet']].to_csv('tw_pitt_cleaned_onlytext.csv', index=False)

# Filter by Word Count 

In [ ]:
# Define word count function
def count_words(wordslst):
    df = pd.DataFrame( {'Word': wordslst, 'Count':np.zeros(len(wordslst))} )
    wordcounts = df.groupby('Word').agg({'Count':np.size}).sort_values(by = 'Count', ascending = False)
    wordcounts['Prop%'] = (wordcounts.Count/len(wordslst))*100
    wordcounts = wordcounts.reset_index()
    
    # sort_values(by=‘ColumnName’, axis=0 , ascending = True , inplace = False, na_position = ‘last’ )
    # the column name of agg must be pre-determined
    # df = df.reset_index() can maintain the original index as a column and add a new index starting from 0

    return wordcounts
def checkWords(word, wordcounts):
    # wordcounts is the df defined from above using count_words( wordslst )     
    return wordcounts[ wordcounts['Word'] == word ]
def display_wordcounts(Type, rank, wordcounts, display=True):
    n = wordcounts.shape[0]
    
    if rank==-1:
        temp = wordcounts
        rank = wordcounts.shape[0]
    elif Type == 'top':
        temp = wordcounts.iloc[:rank,:]
    elif Type == 'bottom':
        temp = wordcounts.iloc[n-rank:,:]
        
    lst = [(temp.iloc[i,0],temp.iloc[i,1],temp.iloc[i,2]) for i in range(rank)]
    
    if display:     
        for i in range(rank):
            print("%-20s\t%-10d\t%.7f"%(lst[i][0],lst[i][1],lst[i][2]))     
    return lst

In [ ]:
tw_pitt = pd.read_csv('tw_pitt_cleaned_onlytext.csv')
tw_pitt = tw_pitt.fillna('')
all_words = ' '.join(list(tw_pitt.cleaned_text)).split()
wordcounts = count_words(all_words)
display(wordcounts)

## Full Text Word Count

In [ ]:
top_prop = 0.0041653
print(wordcounts[wordcounts['Prop%']>=top_prop]['Prop%'].sum())
print(wordcounts[wordcounts['Prop%']>=top_prop]['Count'].sum())
print()
top = display_wordcounts('top', -1, wordcounts[wordcounts['Prop%']>=top_prop])

In [ ]:
bottom_prop = 0.0041653
print(wordcounts[wordcounts['Prop%']<bottom_prop]['Prop%'].sum())
print(wordcounts[wordcounts['Prop%']<bottom_prop]['Count'].sum())
print()
bottom = display_wordcounts('top', -1, wordcounts[wordcounts['Prop%']<bottom_prop])

## Update Stopwords and Filtering

In [ ]:
bottom_words= [x[0] for x in bottom] 

In [ ]:
# add new stopwords
def add_stopwords(wordlst):
    path = workspace('tweets') + 'extra_stopwords_en.txt'   
    with open(path, 'a') as f:
        for word in wordlst:
            f.write('\n'+word)
    f.close()
    return
def update_stopwords():
    from nltk.corpus import stopwords as stpw
    stopwords = []
    path = workspace('tweets')
    with open(path+"stopwords_en.txt", "r") as f1:
        for stopword in f1.readlines():
            stopwords.append(stopword.strip('\n'))
        
    with open(path+"extra_stopwords_en.txt", "r") as f2:
        for stopword in f2.readlines():
            stopwords.append(stopword.strip('\n'))

    stopwords.extend(stpw.words('english'))
    stopwords = set(stopwords)
    return stopwords

In [ ]:
add_stopwords(bottom_words)

In [ ]:
stopwords = update_stopwords()

In [ ]:
def filter_words2(temp):
    # temp = tweet.lower()  
    # fist filter
    temp = [w for w in temp.split(' ') if not w in stopwords]
    temp = [w for w in temp if len(w)>2]
    
    temp = " ".join(word for word in temp)
    return temp

In [ ]:
tw_pitt['cleaned_text'] = tw_pitt.cleaned_text.apply(lambda x: filter_words2(x))

In [ ]:
tw_pitt.to_csv('tw_pitt_cleaned_onlytext.csv', index=False)

# Select by Verb & None

In [ ]:
tw_pitt = pd.read_csv('tw_pitt_cleaned_onlytext.csv')
tw_pitt = tw_pitt.fillna('')

In [ ]:
from nltk.stem.wordnet import WordNetLemmatizer
from nltk import word_tokenize, pos_tag 

def filter_noneNVtag(word2pos):
    # word2pos = ('eating', 'VBG')
    word = word2pos[0]
    tag = word2pos[1]
    
    if tag.startswith('V') or tag.startswith('N'):
        return (word,tag)
    else:
        tag = 'skip'    
    return (word,tag)
def none_and_verb(temp):
    wordlst = word_tokenize(temp)
    word2pos_lst = pos_tag(wordlst)
    word2pos_lst = [filter_noneNVtag(word2pos) for word2pos in word2pos_lst]
    word2pos_lst = [word2pos for word2pos in word2pos_lst if not word2pos[1] == 'skip']
    wordlst = [word2pos[0] for word2pos in word2pos_lst ]
    temp = " ".join(wordlst)
    return temp

def filter_noneNtag(word2pos):
    # word2pos = ('eating', 'VBG')
    word = word2pos[0]
    tag = word2pos[1]
    
    if tag.startswith('N'):
        return (word,tag)
    else:
        tag = 'skip'    
    return (word,tag)
def none(temp):
    wordlst = word_tokenize(temp)
    word2pos_lst = pos_tag(wordlst)
    word2pos_lst = [filter_noneNtag(word2pos) for word2pos in word2pos_lst]
    word2pos_lst = [word2pos for word2pos in word2pos_lst if not word2pos[1] == 'skip']
    wordlst = [word2pos[0] for word2pos in word2pos_lst ]
    temp = " ".join(wordlst)
    return temp

In [ ]:
tw_pitt

In [ ]:
# test function
temp = tw_pitt.iloc[0,1]
print(temp)
print(none_and_verb(temp))
print(none(temp))

In [ ]:
%%time
cleaned_verb_noun = tw_pitt['cleaned_text'].apply(lambda x: none_and_verb(x))
tw_pitt['cleaned_text_verb_noun'] = cleaned_verb_noun

In [ ]:
%%time
cleaned_noun = tw_pitt['cleaned_text'].apply(lambda x: none(x))
tw_pitt['cleaned_text_noun'] = cleaned_noun

In [ ]:
# Extra Filter
tw_pitt['cleaned_text'] = tw_pitt.cleaned_text.apply(lambda x: filter_words2(x))
tw_pitt['cleaned_text_verb_noun'] = tw_pitt.cleaned_text_verb_noun.apply(filter_words2)
tw_pitt['cleaned_text_noun'] = tw_pitt.cleaned_text_noun.apply(filter_words2)
tw_pitt.to_csv('tw_pitt_cleaned_onlytext.csv', index=False)

In [ ]:
original_tw_pitt = tw_pitt[tw_pitt.is_retweet==0]
original_tw_pitt = original_tw_pitt.sort_index()

In [ ]:
original_tw_pitt

In [ ]:
tw_pitt.to_csv('tw_pitt_cleaned_onlytext.csv', index=False)
original_tw_pitt.to_csv('tw_pitt_original_cleaned.csv', index=False)

# Tweets By Day

In [ ]:
tw_pitt = pd.read_csv('tw_pitt_cleaned_onlytext.csv')
tw_pitt = tw_pitt.fillna('')
original_tw_pitt = pd.read_csv('tw_pitt_original_cleaned.csv')
original_tw_pitt = original_tw_pitt.fillna('')

In [ ]:
# Data Type 
tw_pitt_text2day  = tw_pitt[['Date','cleaned_text']]           # All text
tw_pitt_VN2day    = tw_pitt[['Date','cleaned_text_verb_noun']] # All verb and none
tw_pitt_N2day     = tw_pitt[['Date','cleaned_text_noun']]      # All verb and none

original_text2day = original_tw_pitt[['Date','cleaned_text']]            # original text
original_VN2day   = original_tw_pitt[['Date','cleaned_text_verb_noun']]  # original verb and none
original_N2day    = original_tw_pitt[['Date','cleaned_text_noun']]       # original verb and none

In [ ]:
def sumstr(df):
    return ' '.join(list(df.iloc[:,1]))

In [ ]:
tw_pitt_text2day  = tw_pitt_text2day.groupby('Date')[tw_pitt_text2day.columns[1]].agg(' '.join)
tw_pitt_VN2day    = tw_pitt_VN2day.groupby('Date')[tw_pitt_VN2day.columns[1]].agg(' '.join)
tw_pitt_N2day     = tw_pitt_N2day.groupby('Date')[tw_pitt_N2day.columns[1]].agg(' '.join)

original_text2day = original_text2day.groupby('Date')[original_text2day.columns[1]].agg(' '.join)
original_VN2day   = original_VN2day.groupby('Date')[original_VN2day.columns[1]].agg(' '.join)
original_N2day    = original_N2day.groupby('Date')[original_N2day.columns[1]].agg(' '.join)

In [ ]:
tw_pitt_text2day  = pd.DataFrame(tw_pitt_text2day,columns=['Content'])
tw_pitt_VN2day    = pd.DataFrame(tw_pitt_VN2day,columns=['Content'])
tw_pitt_N2day     = pd.DataFrame(tw_pitt_N2day ,columns=['Content'])

original_text2day = pd.DataFrame(original_text2day,columns=['Content'])
original_VN2day   = pd.DataFrame(original_VN2day,columns=['Content'])
original_N2day    = pd.DataFrame(original_N2day,columns=['Content'])

In [ ]:
tw_pitt_text2day.to_csv('Doc1_text2day.csv', index=True)
tw_pitt_VN2day.to_csv('Doc2_VN2day.csv', index=True)
tw_pitt_N2day.to_csv('Doc3_N2day.csv', index=True)

original_text2day.to_csv('Doc4_original_text2day.csv', index=True)
original_VN2day.to_csv('Doc5_original_VN2day.csv', index=True)
original_N2day.to_csv('Doc6_original_N2day.csv', index=True)